# 02_benchmark_models_and_accuracy

**Role.** Sensor/model benchmarking, accuracy diagnostics, and optional spatial robustness checks.

**Pipeline version.** Reproducible scientific pipeline v2 for Dak Lak 2024 coffee mapping Paper 1.


In [1]:
# =============================================================================
# REPRODUCIBILITY BOOTSTRAP: Coffee Paper 1 pipeline v2
# =============================================================================
from pathlib import Path
import os, sys, json, warnings
import numpy as np

# Locate project root robustly whether the notebook is opened from project root
# or from the notebooks/ folder.
_candidate_roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _candidate_roots if (p / "config" / "paper1_config.yaml").exists()), Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from coffeemap.config import load_config, ensure_project_dirs, class_info, class_colors, coffee_class_ids
from coffeemap.manifest import init_run_manifest, append_manifest_note
from coffeemap.plotting import set_publication_style

CONFIG = load_config(PROJECT_ROOT / "config" / "paper1_config.yaml")
PATHS = ensure_project_dirs(CONFIG, PROJECT_ROOT)
CLASS_INFO = class_info(CONFIG)
CLASS_COLORS = class_colors(CONFIG)
CLASS_IDS = sorted(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_IDS]
COFFEE_CLASSES = coffee_class_ids(CONFIG)
RANDOM_SEED = int(CONFIG.get("project", {}).get("random_seed", 42))
np.random.seed(RANDOM_SEED)

TABLES_DIR = PATHS["tables_dir"]
FIGURES_DIR = PATHS["figures_dir"]
SUPPLEMENTARY_DIR = PATHS["supplementary_dir"]
METADATA_DIR = PATHS["metadata_dir"]
INPUT_DIR = PATHS["input_dir"]

NOTEBOOK_NAME = "02_benchmark_models_and_accuracy.ipynb"
MANIFEST = init_run_manifest(CONFIG, PROJECT_ROOT, notebook_name=NOTEBOOK_NAME)
set_publication_style(font="Arial", dpi=600)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Classes: {len(CLASS_IDS)} | Coffee classes: {COFFEE_CLASSES} | Random seed: {RANDOM_SEED}")


Project root: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak
Notebook: 02_benchmark_models_and_accuracy.ipynb
Classes: 10 | Coffee classes: [1, 2, 3] | Random seed: 2024


## Reproducibility contract

This notebook follows the project-level configuration in `config/paper1_config.yaml` and writes outputs only under `results/`.

Key safeguards used in this pipeline:

- class IDs, class names, colors, paths, random seed, and coffee class definitions come from one config file;
- each notebook refreshes `results/metadata/run_manifest.json`;
- feature selection must use training data only;
- validation data are reserved for final assessment;
- Olofsson-style estimates are reported as **area-weighted error-adjusted estimates** unless a mapped-class stratified area-assessment sample is available;
- RF uncertainty is interpreted as **RF vote-based class probability**, not calibrated posterior probability.


In [2]:
# =============================================================================
# Pipeline-level imports commonly used by downstream cells
# =============================================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from coffeemap.io import find_file, read_table, write_table
from coffeemap.schema import (
    detect_column, detect_label_columns, assert_class_ids,
    class_count_table, warn_if_balanced, extract_probability_columns,
    assert_probability_matrix,
)
from coffeemap.metrics import classification_summary, overall_metrics, shannon_entropy, probability_margin
from coffeemap.validation import audit_validation_predictions
from coffeemap.olofsson import error_matrix_counts, area_adjustment, binary_coffee_area_adjustment

SEARCH_DIRS = [INPUT_DIR, PATHS["interim_dir"], TABLES_DIR, SUPPLEMENTARY_DIR, PROJECT_ROOT]
print("Reproducible pipeline helpers loaded.")


Reproducible pipeline helpers loaded.


## A. Table 3: Sensor-ablation cross-validated robustness


In [3]:
"""
03, Table 3: Cross-validated robustness of predictor configurations
=============================================================================
Purpose
-------
This script produces Table 3 as a ROBUSTNESS / SENSOR-ABLATION analysis, not as
replacement for the final independent hold-out accuracy of the mapped product.

The final map should still be reported using the independent 70:30 validation
split. This 5-fold stratified CV is a complementary test showing whether the
ranking of predictor configurations is stable when the training/validation split
changes.

Design
------
1. Combine GEE-exported train and validation sample tables into one labelled pool.
2. For each predictor configuration, run 5-fold stratified CV.
3. Within each fold, perform feature selection ONLY on the training fold:
      RF ranking -> correlation filtering -> max 25 selected predictors.
4. Fit the manuscript-aligned RF and evaluate on the held-out fold.
5. Export fold-level results, manuscript summary, selected features by fold,
   and feature-selection stability.

Inputs
------
  data/Table_TrainSamples_FullFeatureSpace_2024.csv
  data/Table_ValSamples_FullFeatureSpace_2024.csv

Outputs
-------
  Tables/Table3_CV_summary.csv
  Tables/Table3_CV_all_folds.csv
  Tables/Table3_CV_selected_features_by_fold.csv
  Tables/Table3_CV_feature_selection_stability.csv
  Tables/Table3_CV_run_manifest.txt
=============================================================================
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable, List

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold


# -----------------------------------------------------------------------------
# SETTINGS: keep aligned with manuscript and final RF-SHAP workflow
# -----------------------------------------------------------------------------
def find_input_file(filename: str) -> Path:
    """Find GEE-exported input in curated data/ first, then raw GEE export folder."""
    candidates = [
        Path("data/raw") / filename,
        Path("data") / filename,
        Path("GEE_Exports_R3000") / filename,
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Cannot find required input file. Tried:\n  "
        + "\n  ".join(str(p) for p in candidates)
    )


TRAIN_CSV = find_input_file("Table_TrainSamples_FullFeatureSpace_2024.csv")
VAL_CSV = find_input_file("Table_ValSamples_FullFeatureSpace_2024.csv")
LABEL_COL = "class_id"
OUT_DIR = TABLES_DIR

N_FOLDS = 5
RANDOM_STATE = 2024

# Final RF parameters described in the manuscript
RF_TREES = 2000
RF_MAX_FEATURES = 3
RF_MAX_SAMPLES = 0.65
RF_MIN_SAMPLES_LEAF = 1

# Fold-internal feature selection parameters
RFRANK_TREES = 250
CORR_THRESH = 0.90
MAX_FEATURES_FINAL = 25

COFFEE_CLASSES = [1, 2, 3]

CONFIGS = [
    ("Sentinel-2 only", ["S2"]),
    ("Sentinel-1 only", ["S1"]),
    ("Landsat only", ["L89"]),
    ("Sentinel-1 + Sentinel-2", ["S1", "S2"]),
    ("Sentinel-2 + Landsat", ["S2", "L89"]),
    ("Sentinel-1 + Sentinel-2 + DEM", ["S1", "S2", "DEM"]),
    ("Sentinel-1 + Landsat + DEM", ["S1", "L89", "DEM"]),
    ("Sentinel-2 + Landsat + DEM", ["S2", "L89", "DEM"]),
    ("Sentinel-1 + Sentinel-2 + Landsat", ["S1", "S2", "L89"]),
    ("Sentinel-1 + Sentinel-2 + Landsat + DEM", ["S1", "S2", "L89", "DEM"]),
]



# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------
def assert_input_exists(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required input: {path}\n"
            "Run the GEE export first, or place the CSV under the expected data/ folder."
        )


def numeric_feature_columns(df: pd.DataFrame, label_col: str) -> List[str]:
    drop_cols = {
        label_col,
        "system:index",
        ".geo",
        "lon",
        "lat",
        "longitude",
        "latitude",
        "split",
        "gridId",
        "grid_id",
        "rnd",
        "random",
    }
    drop_cols.update(c for c in df.columns if c.startswith("Unnamed"))
    return [
        c
        for c in df.columns
        if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])
    ]


def group_features(cols: Iterable[str]) -> dict[str, List[str]]:
    cols = list(cols)
    return {
        "S2": [c for c in cols if c.startswith("S2_")],
        "S1": [c for c in cols if c.startswith("S1_")],
        "L89": [c for c in cols if c.startswith("L89_")],
        "DEM": [c for c in cols if c.startswith("DEM_")],
    }


def unique_preserve_order(values: Iterable[str]) -> List[str]:
    return list(dict.fromkeys(values))


def features_for_cfg(
    groups_lookup: dict[str, List[str]], group_list: list[str]
) -> List[str]:
    feats: list[str] = []
    for g in group_list:
        feats.extend(groups_lookup.get(g, []))
    return unique_preserve_order(feats)


def rank_rf(
    X_fold: pd.DataFrame, y_fold: pd.Series, feats: list[str], seed: int
) -> list[str]:
    rf = RandomForestClassifier(
        n_estimators=RFRANK_TREES,
        random_state=seed,
        n_jobs=-1,
    )
    rf.fit(X_fold[feats], y_fold)
    imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
    return imp.index.tolist()


def corr_filter(
    X_fold: pd.DataFrame, ranked: list[str], thresh: float, cap: int
) -> list[str]:
    """Greedy correlation filter; applied on training fold only to avoid leakage."""
    if not ranked:
        return []
    corr = X_fold[ranked].corr().abs().fillna(0.0)
    kept: list[str] = []
    for f in ranked:
        if len(kept) >= cap:
            break
        if all(corr.loc[f, s] < thresh for s in kept):
            kept.append(f)
    return kept


def coffee_binary_f1(y_true, y_pred) -> float:
    """Binary coffee-vs-non-coffee F1; coffee classes are 1, 2, and 3."""
    y_true_bin = np.isin(np.asarray(y_true), COFFEE_CLASSES).astype(int)
    y_pred_bin = np.isin(np.asarray(y_pred), COFFEE_CLASSES).astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0)


def coffee_subclass_macro_f1(y_true, y_pred) -> float:
    """
    Macro-F1 for the three coffee production systems across the full fold.

    Unlike the old implementation, this does NOT mask out non-coffee reference
    samples. Therefore false positives from non-coffee into coffee subclasses are
    penalized correctly.
    """
    return f1_score(
        np.asarray(y_true),
        np.asarray(y_pred),
        labels=COFFEE_CLASSES,
        average="macro",
        zero_division=0,
    )


def fmt_mean_std(mean: float, std: float, pct: bool = False, digits: int = 3) -> str:
    if pct:
        return f"{mean * 100:.2f} ± {std * 100:.2f}"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


# -----------------------------------------------------------------------------
# 1) LOAD AND CLEAN
# -----------------------------------------------------------------------------
print("Loading train + validation sample tables...")
assert_input_exists(TRAIN_CSV)
assert_input_exists(VAL_CSV)

tr = pd.read_csv(TRAIN_CSV)
vl = pd.read_csv(VAL_CSV)
print(f"  train: {tr.shape}")
print(f"  val:   {vl.shape}")

combined = pd.concat([tr, vl], ignore_index=True)
if LABEL_COL not in combined.columns:
    raise KeyError(
        f"Label column '{LABEL_COL}' not found. Available columns: {list(combined.columns)[:20]} ..."
    )

feature_cols = numeric_feature_columns(combined, LABEL_COL)
X_all = combined[feature_cols].copy()
y_all = combined[LABEL_COL].copy()

valid = X_all.notnull().all(axis=1) & y_all.notnull()
X_all = X_all.loc[valid].reset_index(drop=True)
y_all = y_all.loc[valid].astype(int).reset_index(drop=True)

print(f"  combined filtered: {X_all.shape}")
print(f"  candidate features: {len(feature_cols)}")
print("  class counts:")
print(y_all.value_counts().sort_index().to_string())

class_counts = y_all.value_counts()
min_class_n = int(class_counts.min()) if not class_counts.empty else 0
if min_class_n < 2:
    raise ValueError(
        "At least one class has fewer than 2 samples after filtering; "
        "stratified CV cannot run."
    )

effective_folds = min(N_FOLDS, min_class_n)
if effective_folds < N_FOLDS:
    print(
        f"Warning: smallest class has {min_class_n} samples; "
        f"reducing folds from {N_FOLDS} to {effective_folds}."
    )

GROUPS = group_features(feature_cols)
print("Features per sensor group:")
for g, fs in GROUPS.items():
    print(f"  {g}: {len(fs)}")

# -----------------------------------------------------------------------------
# 2) CV LOOP
# -----------------------------------------------------------------------------
print(
    f"\nRunning {effective_folds}-fold stratified CV for 10 predictor configurations..."
)
print("Feature selection is repeated inside each training fold to avoid leakage.\n")

skf = StratifiedKFold(n_splits=effective_folds, shuffle=True, random_state=RANDOM_STATE)

all_folds: list[dict] = []
selected_records: list[dict] = []

for config_no, (cfg_name, cfg_groups) in enumerate(CONFIGS, start=1):
    cand_feats = features_for_cfg(GROUPS, cfg_groups)
    if not cand_feats:
        print(f"[{config_no:2d}/10] {cfg_name}: skipped; no candidate features found.")
        continue

    print(f"[{config_no:2d}/10] {cfg_name} ({len(cand_feats)} candidate features)")

    for fold, (tr_idx, te_idx) in enumerate(
        skf.split(X_all[cand_feats], y_all), start=1
    ):
        Xtr, Xte = X_all.iloc[tr_idx], X_all.iloc[te_idx]
        ytr, yte = y_all.iloc[tr_idx], y_all.iloc[te_idx]

        ranked = rank_rf(
            Xtr, ytr, cand_feats, seed=RANDOM_STATE + 100 * config_no + fold
        )
        chosen = corr_filter(Xtr, ranked, CORR_THRESH, MAX_FEATURES_FINAL)
        if not chosen:
            chosen = ranked[:MAX_FEATURES_FINAL]

        for rank, feature in enumerate(chosen, start=1):
            selected_records.append(
                {
                    "config_no": config_no,
                    "config_name": cfg_name,
                    "fold": fold,
                    "rank_after_filter": rank,
                    "feature": feature,
                }
            )

        rf = RandomForestClassifier(
            n_estimators=RF_TREES,
            max_features=RF_MAX_FEATURES,
            max_samples=RF_MAX_SAMPLES,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            bootstrap=True,
            random_state=RANDOM_STATE + 100 * config_no + fold,
            n_jobs=-1,
        )
        rf.fit(Xtr[chosen], ytr)
        ypred = rf.predict(Xte[chosen])

        fold_result = {
            "config_no": config_no,
            "config_name": cfg_name,
            "fold": fold,
            "n_train": len(ytr),
            "n_test": len(yte),
            "n_candidate_features": len(cand_feats),
            "n_selected_features": len(chosen),
            "OA": accuracy_score(yte, ypred),
            "Kappa": cohen_kappa_score(yte, ypred),
            "MacroF1": f1_score(yte, ypred, average="macro", zero_division=0),
            "CoffeeBinaryF1": coffee_binary_f1(yte, ypred),
            "CoffeeSubclassMacroF1": coffee_subclass_macro_f1(yte, ypred),
        }
        all_folds.append(fold_result)

        print(
            f"    fold {fold}: OA={fold_result['OA']:.4f}  "
            f"Kappa={fold_result['Kappa']:.4f}  "
            f"CoffeeBinaryF1={fold_result['CoffeeBinaryF1']:.4f}  "
            f"CoffeeSubclassMacroF1={fold_result['CoffeeSubclassMacroF1']:.4f}"
        )

# -----------------------------------------------------------------------------
# 3) EXPORT FOLD-LEVEL AND FEATURE-SELECTION AUDIT TABLES
# -----------------------------------------------------------------------------
all_folds_df = pd.DataFrame(all_folds)
selected_df = pd.DataFrame(selected_records)

all_folds_path = OUT_DIR / "Table3_CV_all_folds.csv"
selected_path = OUT_DIR / "Table3_CV_selected_features_by_fold.csv"
all_folds_df.to_csv(all_folds_path, index=False, encoding="utf-8-sig")
selected_df.to_csv(selected_path, index=False, encoding="utf-8-sig")

if not selected_df.empty:
    stability = (
        selected_df.groupby(["config_no", "config_name", "feature"], sort=False)
        .agg(
            selected_in_n_folds=("fold", "nunique"),
            mean_rank_after_filter=("rank_after_filter", "mean"),
            min_rank_after_filter=("rank_after_filter", "min"),
            max_rank_after_filter=("rank_after_filter", "max"),
        )
        .reset_index()
        .sort_values(
            ["config_no", "selected_in_n_folds", "mean_rank_after_filter"],
            ascending=[True, False, True],
        )
    )
else:
    stability = pd.DataFrame()

stability_path = OUT_DIR / "Table3_CV_feature_selection_stability.csv"
stability.to_csv(stability_path, index=False, encoding="utf-8-sig")

# -----------------------------------------------------------------------------
# 4) AGGREGATE MANUSCRIPT TABLE
# -----------------------------------------------------------------------------
if all_folds_df.empty:
    summary = pd.DataFrame(
        columns=[
            "No",
            "Predictor configuration",
            "Mean selected predictors",
            "OA (%)",
            "Kappa",
            "Macro F1",
            "Coffee binary F1",
            "Coffee subclass macro F1",
        ]
    )
else:
    agg = (
        all_folds_df.groupby(["config_no", "config_name"], sort=False)
        .agg(
            n_selected_features=("n_selected_features", "mean"),
            OA_mean=("OA", "mean"),
            OA_std=("OA", "std"),
            Kappa_mean=("Kappa", "mean"),
            Kappa_std=("Kappa", "std"),
            MacroF1_mean=("MacroF1", "mean"),
            MacroF1_std=("MacroF1", "std"),
            CoffeeBinaryF1_mean=("CoffeeBinaryF1", "mean"),
            CoffeeBinaryF1_std=("CoffeeBinaryF1", "std"),
            CoffeeSubclassMacroF1_mean=("CoffeeSubclassMacroF1", "mean"),
            CoffeeSubclassMacroF1_std=("CoffeeSubclassMacroF1", "std"),
        )
        .reset_index()
    )

    agg["OA (%)"] = agg.apply(
        lambda r: fmt_mean_std(r.OA_mean, r.OA_std, pct=True), axis=1
    )
    agg["Kappa"] = agg.apply(lambda r: fmt_mean_std(r.Kappa_mean, r.Kappa_std), axis=1)
    agg["Macro F1"] = agg.apply(
        lambda r: fmt_mean_std(r.MacroF1_mean, r.MacroF1_std), axis=1
    )
    agg["Coffee binary F1"] = agg.apply(
        lambda r: fmt_mean_std(r.CoffeeBinaryF1_mean, r.CoffeeBinaryF1_std), axis=1
    )
    agg["Coffee subclass macro F1"] = agg.apply(
        lambda r: fmt_mean_std(
            r.CoffeeSubclassMacroF1_mean, r.CoffeeSubclassMacroF1_std
        ),
        axis=1,
    )

    summary = agg[
        [
            "config_no",
            "config_name",
            "n_selected_features",
            "OA (%)",
            "Kappa",
            "Macro F1",
            "Coffee binary F1",
            "Coffee subclass macro F1",
        ]
    ].copy()
    summary.columns = [
        "No",
        "Predictor configuration",
        "Mean selected predictors",
        "OA (%)",
        "Kappa",
        "Macro F1",
        "Coffee binary F1",
        "Coffee subclass macro F1",
    ]
    summary["Mean selected predictors"] = summary["Mean selected predictors"].round(1)

summary_path = OUT_DIR / "Table3_CV_summary.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

# Excel workbook for manuscript/table audit
excel_path = OUT_DIR / "Table3_CV_summary.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Table3_summary", index=False)
    all_folds_df.to_excel(writer, sheet_name="All_folds", index=False)
    stability.to_excel(writer, sheet_name="Feature_stability", index=False)

# -----------------------------------------------------------------------------
# 5) RUN MANIFEST
# -----------------------------------------------------------------------------
manifest_path = OUT_DIR / "Table3_CV_run_manifest.txt"
with open(manifest_path, "w", encoding="utf-8") as f:
    f.write("Table 3 cross-validated robustness run manifest\n")
    f.write("================================================\n")
    f.write("Important interpretation:\n")
    f.write(
        "  This table is a sensor-ablation / robustness analysis. It should not replace\n"
    )
    f.write(
        "  the independent hold-out accuracy used to report the final mapped product.\n\n"
    )
    f.write(f"Train CSV: {TRAIN_CSV}\n")
    f.write(f"Validation CSV: {VAL_CSV}\n")
    f.write(f"Combined valid samples: {len(y_all)}\n")
    f.write(f"Candidate numeric features: {len(feature_cols)}\n")
    f.write(f"Requested folds: {N_FOLDS}\n")
    f.write(f"Effective folds: {effective_folds}\n")
    f.write(f"Random state: {RANDOM_STATE}\n")
    f.write(f"RF trees: {RF_TREES}\n")
    f.write(f"RF max_features: {RF_MAX_FEATURES}\n")
    f.write(f"RF max_samples: {RF_MAX_SAMPLES}\n")
    f.write(f"RF min_samples_leaf: {RF_MIN_SAMPLES_LEAF}\n")
    f.write(f"Ranking RF trees: {RFRANK_TREES}\n")
    f.write(f"Correlation threshold: {CORR_THRESH}\n")
    f.write(f"Max final features: {MAX_FEATURES_FINAL}\n\n")
    f.write("Feature counts per sensor group:\n")
    for g, fs in GROUPS.items():
        f.write(f"  {g}: {len(fs)}\n")
    f.write("\nClass counts:\n")
    f.write(y_all.value_counts().sort_index().to_string())
    f.write("\n")

print("\nTable 3 CV summary:\n")
print(summary.to_string(index=False))
print("\nSaved outputs:")
for p in [
    summary_path,
    excel_path,
    all_folds_path,
    selected_path,
    stability_path,
    manifest_path,
]:
    print(f"  {p}")


Loading train + validation sample tables...


  train: (2100, 100)
  val:   (900, 100)
  combined filtered: (3000, 97)
  candidate features: 97
  class counts:
class_id
1     300
2     300
3     300
4     300
5     300
6     300
7     300
8     300
9     300
10    300
Features per sensor group:
  S2: 52
  S1: 15
  L89: 26
  DEM: 4

Running 5-fold stratified CV for 10 predictor configurations...
Feature selection is repeated inside each training fold to avoid leakage.

[ 1/10] Sentinel-2 only (52 candidate features)


    fold 1: OA=0.9250  Kappa=0.9167  CoffeeBinaryF1=0.9385  CoffeeSubclassMacroF1=0.8533


    fold 2: OA=0.9333  Kappa=0.9259  CoffeeBinaryF1=0.9687  CoffeeSubclassMacroF1=0.8448


    fold 3: OA=0.9200  Kappa=0.9111  CoffeeBinaryF1=0.9477  CoffeeSubclassMacroF1=0.8636


...

    fold 2: OA=0.9483  Kappa=0.9426  CoffeeBinaryF1=0.9716  CoffeeSubclassMacroF1=0.8721


    fold 3: OA=0.9617  Kappa=0.9574  CoffeeBinaryF1=0.9809  CoffeeSubclassMacroF1=0.9320


    fold 4: OA=0.9733  Kappa=0.9704  CoffeeBinaryF1=0.9805  CoffeeSubclassMacroF1=0.9351


    fold 5: OA=0.9467  Kappa=0.9407  CoffeeBinaryF1=0.9697  CoffeeSubclassMacroF1=0.8740



Table 3 CV summary:

 No                 Predictor configuration  Mean selected predictors       OA (%)         Kappa      Macro F1 Coffee binary F1 Coffee subclass macro F1
  1                         Sentinel-2 only                      25.0 92.53 ± 1.01 0.917 ± 0.011 0.924 ± 0.010    0.956 ± 0.013            0.851 ± 0.023
  2                         Sentinel-1 only                      13.0 65.13 ± 1.51 0.613 ± 0.017 0.648 ± 0.016    0.684 ± 0.033            0.515 ± 0.039
  3                            Landsat only                      16.8 92.40 ± 0.98 0.916 ± 0.011 0.924 ± 0.010    0.953 ± 0.011            0.882 ± 0.011
  4                 Sentinel-1 + Sentinel-2                      25.0 92.93 ± 1.18 0.921 ± 0.013 0.928 ± 0.012    0.961 ± 0.008            0.847 ± 0.032
  5                    Sentinel-2 + Landsat                      25.0 93.97 ± 0.92 0.933 ± 0.010 0.939 ± 0.010    0.969 ± 0.007            0.874 ± 0.027
  6           Sentinel-1 + Sentinel-2 + DEM                 

In [4]:
# =============================================================================
# 3B. Table 3 significance test: full multi-sensor model vs. each single-sensor
# config, Holm-corrected pairwise Wilcoxon signed-rank tests across the 5 CV folds
# =============================================================================
from scipy.stats import wilcoxon

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(n, dtype=float)
    prev = 0
    for rank, idx in enumerate(order):
        adj = (n - rank) * p_values[idx]
        adj = max(adj, prev)
        adjusted[idx] = min(adj, 1.0)
        prev = adjusted[idx]
    return adjusted

FULL_CONFIG = "Sentinel-1 + Sentinel-2 + Landsat + DEM"
SINGLE_SENSOR_CONFIGS = ["Sentinel-2 only", "Sentinel-1 only", "Landsat only"]
sig_metrics = {"OA": "Overall accuracy", "MacroF1": "Macro F1"}

sig_rows = []
for metric, metric_label in sig_metrics.items():
    wide = all_folds_df.pivot_table(index="fold", columns="config_name", values=metric)
    raw_p = []
    raw_rows = []
    for cfg in SINGLE_SENSOR_CONFIGS:
        x = wide[FULL_CONFIG].values
        y = wide[cfg].values
        if len(x) >= 3 and not np.allclose(x, y):
            w_stat, p_pair = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
        else:
            w_stat, p_pair = np.nan, np.nan
        raw_rows.append({
            "metric": metric_label,
            "comparison": f"{FULL_CONFIG} vs {cfg}",
            "n_folds": len(x),
            "mean_difference": float(np.mean(x - y)),
            "wilcoxon_statistic": w_stat,
            "p_value_raw": p_pair,
        })
        raw_p.append(p_pair)
    adj_p = holm_adjust(np.asarray(raw_p, dtype=float))
    for row, p_adj in zip(raw_rows, adj_p):
        row["p_value_holm"] = p_adj
        sig_rows.append(row)

table3_sig_df = pd.DataFrame(sig_rows)
table3_sig_path = OUT_DIR / "Table3_Statistical_Comparison.csv"
table3_sig_df.to_csv(table3_sig_path, index=False, encoding="utf-8-sig")
print("Saved:", table3_sig_path)
print(table3_sig_df.to_string(index=False))


Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table3_Statistical_Comparison.csv
          metric                                                 comparison  n_folds  mean_difference  wilcoxon_statistic  p_value_raw  p_value_holm
Overall accuracy Sentinel-1 + Sentinel-2 + Landsat + DEM vs Sentinel-2 only        5         0.033667                 0.0       0.0625        0.1875
Overall accuracy Sentinel-1 + Sentinel-2 + Landsat + DEM vs Sentinel-1 only        5         0.307667                 0.0       0.0625        0.1875
Overall accuracy    Sentinel-1 + Sentinel-2 + Landsat + DEM vs Landsat only        5         0.035000                 0.0       0.0625        0.1875
        Macro F1 Sentinel-1 + Sentinel-2 + Landsat + DEM vs Sentinel-2 only        5         0.034340                 0.0       0.0625        0.1875
        Macro F1 Sentinel-1 + Sentinel-2 + Landsat + DEM vs Sentinel-1 only        5         0.310278                 0.0       0.0625       

## B. Table 4: Class-wise accuracy for the final selected model


In [5]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Search order: raw GEE exports first, then curated data/, then downstream outputs.
IN_DIRS = [
    Path("data/raw"),
    Path("data/raw/data_DakLak_Statistics"),
    Path("GEE_Exports_R3000"),
    Path("data"),
    Path("Supplementary"),
    Path("Figures"),
    Path("."),
]
OUT_DIR = TABLES_DIR

CLASS_INFO = {
    1: "Sun coffee",
    2: "Intercrop coffee",
    3: "Newly planted coffee",
    4: "Rubber",
    5: "Partially vegetative",
    6: "Rice",
    7: "Other upland crops",
    8: "Forest",
    9: "Water",
    10: "Built",
}
CLASS_ORDER = list(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_ORDER]
ROUND_DIGITS = 1
EXPORT_PREFIX = "Table4_classwise_accuracy_best_model"
print("Output folder:", OUT_DIR.resolve())


Output folder: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables


## 1. Locate validation predictions or confusion matrix

Preferred input is a validation-prediction CSV with one column for the reference label and one column for the predicted label. The notebook tries common file names and column names automatically. If your file has a different name, edit `MANUAL_INPUT_FILE` below.

In [6]:
# Set this only if you want to force a specific file.
# Preferred final input from GEE v4:
# MANUAL_INPUT_FILE = "GEE_Exports_R3000/Table_ValPredictions_RF_Final_2024.csv"
MANUAL_INPUT_FILE = None

PREFERRED_PREDICTION_FILENAMES = [
    "Table_ValPredictions_RF_Final_2024.csv",
    "Table_ValPredictions_RF_Final_2024_FINALv2.csv",
]

PREDICTION_FILE_PATTERNS = [
    "*ValPredictions*RF*Final*.csv",
    "*validation*prediction*.csv", "*val*prediction*.csv", "*valid*prediction*.csv",
    "*test*prediction*.csv", "*classified*validation*.csv", "*predictions*.csv",
]
CONFUSION_FILE_PATTERNS = [
    "*ConfusionMatrix*RowNorm*csv",
    "*ConfusionMatrix*Long*csv",
    "*confusion*matrix*.csv", "*confusion*.csv", "*matrix*.csv",
]

def find_candidates(patterns):
    out = []
    for d in IN_DIRS:
        if not d.exists():
            continue
        for pat in patterns:
            out.extend(sorted(d.glob(pat)))
    # de-duplicate while preserving order
    seen = []
    for p in out:
        if p not in seen:
            seen.append(p)
    return seen

def preferred_prediction_candidates():
    preferred = []
    for d in IN_DIRS:
        if not d.exists():
            continue
        for fn in PREFERRED_PREDICTION_FILENAMES:
            p = d / fn
            if p.exists():
                preferred.append(p)
    rest = find_candidates(PREDICTION_FILE_PATTERNS)
    out = []
    for p in preferred + rest:
        if p not in out:
            out.append(p)
    return out

prediction_candidates = preferred_prediction_candidates()
confusion_candidates = find_candidates(CONFUSION_FILE_PATTERNS)

print("Prediction candidates:")
for i, p in enumerate(prediction_candidates[:20], 1):
    print(f"  {i:02d}. {p}")
print("\nConfusion-matrix candidates:")
for i, p in enumerate(confusion_candidates[:20], 1):
    print(f"  {i:02d}. {p}")


Prediction candidates:
  01. data\raw\Table_ValPredictions_RF_Final_2024.csv
  02. data\raw\Table_ValPredictions_RF_FullFeatures_2024.csv

Confusion-matrix candidates:
  01. data\raw\Table_ConfusionMatrix_Long_DakLak2024_corrTop25.csv
  02. data\raw\Table_ConfusionMatrix_Long_DakLak2024_fullFeatures.csv


## 2. Helper functions

In [7]:
TRUE_COL_CANDIDATES = [
    "class_id", "true_class_id", "true_class", "reference", "actual", "truth", "observed",
    "label", "y_true", "ref", "class",
]
PRED_COL_CANDIDATES = [
    "classification", "pred_class_id", "pred_class", "predicted", "prediction", "pred",
    "classified", "rf_prediction", "y_pred", "model_pred", "classification_result",
]

def normalise_col(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def detect_label_columns(df):
    norm_to_original = {normalise_col(c): c for c in df.columns}
    true_col = None
    pred_col = None
    for cand in TRUE_COL_CANDIDATES:
        if normalise_col(cand) in norm_to_original:
            true_col = norm_to_original[normalise_col(cand)]
            break
    for cand in PRED_COL_CANDIDATES:
        if normalise_col(cand) in norm_to_original:
            pred_col = norm_to_original[normalise_col(cand)]
            break
    if true_col is None or pred_col is None:
        numeric_cols = []
        for c in df.columns:
            vals = pd.to_numeric(df[c], errors="coerce").dropna()
            if len(vals) == 0:
                continue
            unique = set(vals.astype(int).unique())
            if unique and unique.issubset(set(CLASS_ORDER)):
                numeric_cols.append(c)
        if true_col is None and len(numeric_cols) >= 1:
            true_col = numeric_cols[0]
        if pred_col is None and len(numeric_cols) >= 2:
            pred_col = numeric_cols[1]
    if true_col is None or pred_col is None:
        raise ValueError(f"Could not detect true/predicted label columns. Available columns: {list(df.columns)}")
    if true_col == pred_col:
        raise ValueError(f"Detected the same column for true and predicted labels: {true_col}")
    return true_col, pred_col

def confusion_from_predictions(df, true_col=None, pred_col=None):
    if true_col is None or pred_col is None:
        true_col, pred_col = detect_label_columns(df)
    y_true = pd.to_numeric(df[true_col], errors="coerce")
    y_pred = pd.to_numeric(df[pred_col], errors="coerce")
    valid = y_true.notna() & y_pred.notna()
    y_true = y_true[valid].astype(int)
    y_pred = y_pred[valid].astype(int)
    cm = pd.crosstab(y_true, y_pred, rownames=["Reference"], colnames=["Predicted"], dropna=False)
    cm = cm.reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)
    return cm, true_col, pred_col, len(y_true)

def read_confusion_matrix(path):
    raw = pd.read_csv(path)

    # GEE long-format raw confusion matrix.
    if "count" in raw.columns:
        true_col = "true_class_id" if "true_class_id" in raw.columns else "true_class"
        pred_col = "pred_class_id" if "pred_class_id" in raw.columns else "pred_class"
        if true_col not in raw.columns or pred_col not in raw.columns:
            raise ValueError(f"Long confusion matrix must include true/pred class columns. Found: {raw.columns.tolist()}")
        cm = np.zeros((len(CLASS_ORDER), len(CLASS_ORDER)), dtype=int)
        for _, r in raw.iterrows():
            t, p, c = int(r[true_col]), int(r[pred_col]), int(r["count"])
            if t in CLASS_ORDER and p in CLASS_ORDER:
                cm[CLASS_ORDER.index(t), CLASS_ORDER.index(p)] = c
        return pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER).rename_axis("Reference").rename_axis("Predicted", axis=1)

    # Square matrix CSV.
    df = raw.copy()
    first_col = str(df.columns[0]).lower()
    if first_col in ["reference", "actual", "class", "class_id", "label", "unnamed: 0"] or not str(df.columns[0]).isdigit():
        idx = df.iloc[:, 0]
        body = df.iloc[:, 1:].copy()
        try:
            body.index = pd.to_numeric(idx).astype(int)
        except Exception:
            name_to_id = {v.lower(): k for k, v in CLASS_INFO.items()}
            body.index = [name_to_id.get(str(v).lower(), v) for v in idx]
        df = body
    df.columns = [int(float(c)) if str(c).replace(".", "", 1).isdigit() else c for c in df.columns]
    df.index = [int(float(i)) if str(i).replace(".", "", 1).isdigit() else i for i in df.index]
    cm = df.apply(pd.to_numeric, errors="coerce").fillna(0)
    cm = cm.reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0).astype(int)
    cm.index.name = "Reference"
    cm.columns.name = "Predicted"
    return cm

from statsmodels.stats.proportion import proportion_confint

def metrics_from_cm(cm):
    cm = cm.reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0).astype(float)
    tp = np.diag(cm.values)
    row_sum = cm.sum(axis=1).values
    col_sum = cm.sum(axis=0).values
    total = cm.values.sum()
    producer = np.divide(tp, row_sum, out=np.zeros_like(tp), where=row_sum != 0)
    user = np.divide(tp, col_sum, out=np.zeros_like(tp), where=col_sum != 0)
    f1 = np.divide(2 * producer * user, producer + user, out=np.zeros_like(tp), where=(producer + user) != 0)
    # Wilson score 95% CI on the naive (unadjusted) per-class PA/UA proportions.
    # Distinct from the Olofsson area-weighted CI reported for Table 6 / Supplementary Table S5.
    pa_lo, pa_hi = proportion_confint(tp, row_sum, method="wilson")
    ua_lo, ua_hi = proportion_confint(tp, col_sum, method="wilson")
    table = pd.DataFrame({
        "Class ID": CLASS_ORDER,
        "Class": CLASS_NAMES,
        "Validation samples": row_sum.astype(int),
        "Producer's accuracy (%)": producer * 100,
        "Producer's accuracy 95% CI lower (%)": pa_lo * 100,
        "Producer's accuracy 95% CI upper (%)": pa_hi * 100,
        "User's accuracy (%)": user * 100,
        "User's accuracy 95% CI lower (%)": ua_lo * 100,
        "User's accuracy 95% CI upper (%)": ua_hi * 100,
        "F1-score": f1,
    })
    oa = tp.sum() / total if total else np.nan
    macro_f1 = np.nanmean(f1)
    coffee_idx = [0, 1, 2]
    coffee_support = row_sum[coffee_idx].sum()
    coffee_f1_weighted = np.average(f1[coffee_idx], weights=row_sum[coffee_idx]) if coffee_support else np.nan
    summary = {
        "Total validation samples": int(total),
        "Overall accuracy": oa,
        "Macro F1": macro_f1,
        "Coffee weighted F1": coffee_f1_weighted,
    }
    return table, summary

def format_table_for_manuscript(table):
    out = table.copy()
    for col in [
        "Producer's accuracy (%)", "Producer's accuracy 95% CI lower (%)", "Producer's accuracy 95% CI upper (%)",
        "User's accuracy (%)", "User's accuracy 95% CI lower (%)", "User's accuracy 95% CI upper (%)",
    ]:
        out[col] = out[col].round(ROUND_DIGITS)
    out["F1-score"] = out["F1-score"].round(3)
    return out


## 3. Load input and compute Table 4

In [8]:
if MANUAL_INPUT_FILE is not None:
    input_path = Path(MANUAL_INPUT_FILE)
else:
    input_path = prediction_candidates[0] if prediction_candidates else None

cm = None
if input_path is not None and input_path.exists():
    df_pred = pd.read_csv(input_path)
    try:
        cm, true_col, pred_col, n_valid = confusion_from_predictions(df_pred)
        print(f"Loaded validation predictions: {input_path}")
        print(f"Detected true column: {true_col}")
        print(f"Detected predicted column: {pred_col}")
        print(f"Valid label pairs: {n_valid}")
    except Exception as e:
        print("Could not use prediction file:", e)
        cm = None

if cm is None:
    if confusion_candidates:
        input_path = confusion_candidates[0]
        cm = read_confusion_matrix(input_path)
        print(f"Loaded confusion matrix: {input_path}")
    else:
        raise FileNotFoundError("No validation prediction or confusion-matrix CSV found. Set MANUAL_INPUT_FILE.")

print("\nConfusion matrix: rows = reference, columns = predicted")
display(cm)

table4_raw, performance_summary = metrics_from_cm(cm)
table4 = format_table_for_manuscript(table4_raw)
print("\nPerformance summary:")
for k, v in performance_summary.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
print("\nTable 4:")
display(table4)


Loaded validation predictions: data\raw\Table_ValPredictions_RF_Final_2024.csv
Detected true column: class_id
Detected predicted column: classification
Valid label pairs: 900

Confusion matrix: rows = reference, columns = predicted


Predicted,1,2,3,4,5,6,7,8,9,10
Reference,,,,,,,,,,
1,72,16,0,0,0,0,0,2,0,0
2,4,83,0,1,1,0,0,1,0,0
3,1,0,87,0,1,0,0,0,0,1
4,3,0,0,82,1,0,0,4,0,0
5,1,0,2,0,87,0,0,0,0,0
6,0,0,0,0,0,82,3,0,1,4
7,0,0,0,0,2,0,86,2,0,0
8,0,0,0,0,2,0,0,88,0,0
9,0,0,0,0,0,0,2,0,87,1



Performance summary:
Total validation samples: 900
Overall accuracy: 0.9344
Macro F1: 0.9343
Coffee weighted F1: 0.8975

Table 4:


,Class ID,Class,Validation samples,Producer's accuracy (%),Producer's accuracy 95% CI lower (%),Producer's accuracy 95% CI upper (%),User's accuracy (%),User's accuracy 95% CI lower (%),User's accuracy 95% CI upper (%),F1-score
0,1,Sun coffee,90,80.0,70.6,87.0,88.9,80.2,94.0,0.842
1,2,Intercrop coffee,90,92.2,84.8,96.2,83.8,75.3,89.8,0.878
2,3,Newly planted coffee,90,96.7,90.7,98.9,97.8,92.2,99.4,0.972
3,4,Rubber,90,91.1,83.4,95.4,98.8,93.5,99.8,0.948
4,5,Partially vegetative,90,96.7,90.7,98.9,92.6,85.4,96.3,0.946
5,6,Rice,90,91.1,83.4,95.4,100.0,95.5,100.0,0.953
6,7,Other upland crops,90,95.6,89.1,98.3,94.5,87.8,97.6,0.950
7,8,Forest,90,97.8,92.3,99.4,90.7,83.3,95.0,0.941
8,9,Water,90,96.7,90.7,98.9,95.6,89.2,98.3,0.961
9,10,Built,90,96.7,90.7,98.9,93.5,86.6,97.0,0.951


## 4. Export Table 4

In [9]:
from importlib.util import find_spec

csv_path = OUT_DIR / f"{EXPORT_PREFIX}.csv"
xlsx_path = OUT_DIR / f"{EXPORT_PREFIX}.xlsx"
md_path = OUT_DIR / f"{EXPORT_PREFIX}.md"
tex_path = OUT_DIR / f"{EXPORT_PREFIX}.tex"
cm_path = OUT_DIR / f"{EXPORT_PREFIX}_confusion_matrix_used.csv"
summary_path = OUT_DIR / f"{EXPORT_PREFIX}_summary.csv"


def write_excel_workbook(path, table, confusion_matrix, summary):
    engine = None
    if find_spec("xlsxwriter") is not None:
        engine = "xlsxwriter"
    elif find_spec("openpyxl") is not None:
        engine = "openpyxl"
    else:
        return None

    with pd.ExcelWriter(path, engine=engine) as writer:
        table.to_excel(writer, sheet_name="Table4", index=False)
        confusion_matrix.to_excel(writer, sheet_name="Confusion_matrix")
        pd.DataFrame([summary]).to_excel(writer, sheet_name="Summary", index=False)

        if engine == "xlsxwriter":
            workbook = writer.book
            header_fmt = workbook.add_format({"bold": True, "text_wrap": True, "valign": "middle", "align": "center", "bg_color": "#EAF2F8", "border": 1})
            body_fmt = workbook.add_format({"valign": "middle", "border": 1})
            num_fmt = workbook.add_format({"valign": "middle", "border": 1, "num_format": "0.0"})
            f1_fmt = workbook.add_format({"valign": "middle", "border": 1, "num_format": "0.000"})
            ws = writer.sheets["Table4"]
            ws.freeze_panes(1, 0)
            for i, w in enumerate([9, 24, 18, 24, 20, 10]):
                ws.set_column(i, i, w)
            for col_num, value in enumerate(table.columns.values):
                ws.write(0, col_num, value, header_fmt)
            for row in range(1, len(table) + 1):
                ws.set_row(row, 21)
                for col in range(len(table.columns)):
                    value = table.iloc[row - 1, col]
                    fmt = num_fmt if col in [3, 4] else f1_fmt if col == 5 else body_fmt
                    ws.write(row, col, value, fmt)
            writer.sheets["Confusion_matrix"].freeze_panes(1, 1)

    return engine


def dataframe_to_markdown(df):
    def format_cell(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return f"{value:.3f}" if abs(value) <= 1 else f"{value:.1f}"
        return str(value)

    headers = [str(col) for col in df.columns]
    rows = [[format_cell(value) for value in row] for row in df.itertuples(index=False, name=None)]
    widths = [len(header) for header in headers]
    for row in rows:
        for idx, value in enumerate(row):
            widths[idx] = max(widths[idx], len(value))

    def render_row(values):
        return "| " + " | ".join(value.ljust(widths[idx]) for idx, value in enumerate(values)) + " |"

    align = [":" + "-" * max(width - 2, 1) + ":" if width > 2 else ":-:" for width in widths]
    lines = [render_row(headers), "| " + " | ".join(align) + " |"]
    lines.extend(render_row(row) for row in rows)
    return "\n".join(lines)


xlsx_engine = write_excel_workbook(xlsx_path, table4, cm, performance_summary)
output_paths = [csv_path, md_path, tex_path, cm_path, summary_path]
if xlsx_engine is not None:
    output_paths.insert(1, xlsx_path)
else:
    print("Excel export skipped: install either xlsxwriter or openpyxl to create the XLSX output.")

table4.to_csv(csv_path, index=False)
cm.to_csv(cm_path)
pd.DataFrame([performance_summary]).to_csv(summary_path, index=False)

md_text = dataframe_to_markdown(table4)
md_path.write_text(md_text, encoding="utf-8")
latex_text = table4.to_latex(index=False, escape=True, float_format=lambda x: f"{x:.3f}" if abs(x) <= 1 else f"{x:.1f}")
tex_path.write_text(latex_text, encoding="utf-8")

print("Saved outputs:")
for p in output_paths:
    print(" -", p)


Saved outputs:
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table4_classwise_accuracy_best_model.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table4_classwise_accuracy_best_model.xlsx
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table4_classwise_accuracy_best_model.md
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table4_classwise_accuracy_best_model.tex
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table4_classwise_accuracy_best_model_confusion_matrix_used.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\tables\Table4_classwise_accuracy_best_model_summary.csv


## 5. Manuscript-ready caption and Excel table

This step prints the table caption and exports an additional manuscript-ready Excel file containing only Table 4 plus a short caption sheet.


In [10]:
caption = """Table 4. Class-wise producer's accuracy, user's accuracy, and F1-score for the best-performing Random Forest model, independent 900-point held-out validation set (n=90 per class). Producer's accuracy represents omission-error sensitivity for each reference class, whereas user's accuracy represents commission-error sensitivity for each mapped class. F1-score is the harmonic mean of producer's and user's accuracy. 95% CIs are Wilson score intervals on the naive per-class proportions."""

manuscript_xlsx_path = OUT_DIR / f"{EXPORT_PREFIX}_manuscript_ready.xlsx"

def write_manuscript_excel(path, table, caption_text):
    engine = None
    if find_spec("xlsxwriter") is not None:
        engine = "xlsxwriter"
    elif find_spec("openpyxl") is not None:
        engine = "openpyxl"
    else:
        return None

    with pd.ExcelWriter(path, engine=engine) as writer:
        table.to_excel(writer, sheet_name="Table4_manuscript", index=False)
        pd.DataFrame({"Caption": [caption_text]}).to_excel(writer, sheet_name="Caption", index=False)

        if engine == "xlsxwriter":
            workbook = writer.book
            ws = writer.sheets["Table4_manuscript"]
            header_fmt = workbook.add_format({
                "bold": True,
                "text_wrap": True,
                "valign": "middle",
                "align": "center",
                "bg_color": "#EAF2F8",
                "border": 1
            })
            body_fmt = workbook.add_format({"valign": "middle", "border": 1})
            int_fmt = workbook.add_format({"valign": "middle", "align": "center", "border": 1, "num_format": "0"})
            acc_fmt = workbook.add_format({"valign": "middle", "align": "center", "border": 1, "num_format": "0.0"})
            f1_fmt = workbook.add_format({"valign": "middle", "align": "center", "border": 1, "num_format": "0.000"})

            ws.freeze_panes(1, 0)
            widths = {
                0: 9,
                1: 26,
                2: 18,
                3: 24,
                4: 22,
                5: 12,
            }
            for col_idx, width in widths.items():
                ws.set_column(col_idx, col_idx, width)

            ws.set_row(0, 34)
            for col_num, value in enumerate(table.columns.values):
                ws.write(0, col_num, value, header_fmt)

            for row in range(1, len(table) + 1):
                ws.set_row(row, 22)
                for col in range(len(table.columns)):
                    value = table.iloc[row - 1, col]
                    if col in [0, 2]:
                        fmt = int_fmt
                    elif col in [3, 4]:
                        fmt = acc_fmt
                    elif col == 5:
                        fmt = f1_fmt
                    else:
                        fmt = body_fmt
                    ws.write(row, col, value, fmt)

            ws_caption = writer.sheets["Caption"]
            caption_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
            ws_caption.set_column(0, 0, 120)
            ws_caption.set_row(1, 90)
            ws_caption.write(0, 0, "Caption", header_fmt)
            ws_caption.write(1, 0, caption_text, caption_fmt)

    return engine

engine = write_manuscript_excel(manuscript_xlsx_path, table4, caption)

print(caption)
print("\nMarkdown table:\n")
print(md_text)

if engine is not None:
    print("\nSaved manuscript-ready Excel table:")
    print(" -", manuscript_xlsx_path)
else:
    print("\nExcel manuscript export skipped: install either xlsxwriter or openpyxl.")


Table 4. Class-wise producer's accuracy, user's accuracy, and F1-score for the best-performing Random Forest model, independent 900-point held-out validation set (n=90 per class). Producer's accuracy represents omission-error sensitivity for each reference class, whereas user's accuracy represents commission-error sensitivity for each mapped class. F1-score is the harmonic mean of producer's and user's accuracy. 95% CIs are Wilson score intervals on the naive per-class proportions.



Markdown table:

| Class ID | Class                | Validation samples | Producer's accuracy (%) | Producer's accuracy 95% CI lower (%) | Producer's accuracy 95% CI upper (%) | User's accuracy (%) | User's accuracy 95% CI lower (%) | User's accuracy 95% CI upper (%) | F1-score |
| :------: | :------------------: | :----------------: | :---------------------: | :----------------------------------: | :----------------------------------: | :-----------------: | :------------------------------: | :------------------------------: | :------: |
| 1        | Sun coffee           | 90                 | 80.0                    | 70.6                                 | 87.0                                 | 88.9                | 80.2                             | 94.0                             | 0.842    |
| 2        | Intercrop coffee     | 90                 | 92.2                    | 84.8                                 | 96.2                                 | 83.8                | 75.3  

## C. Figure 4: Row-normalized confusion matrix


In [11]:
# -*- coding: utf-8 -*-
"""
04, Figure 4: Row-normalized confusion matrix for the best-performing model
=============================================================================
Robust upgraded version.

Accepts any of the following inputs:
  - GEE row-normalized long table: true_class_id, pred_class_id, row_normalized_percent
  - GEE raw long table: true_class_id/pred_class_id/count or true_class/pred_class/count
  - Python square confusion matrix CSV.

Outputs:
  Figures/Figure5_ConfusionMatrix.png/pdf/svg
  Figures/fig4_producer_accuracy.csv
=============================================================================
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

CANDIDATE_INPUTS = [
    Path("data/raw/Table_ConfusionMatrix_Long_DakLak2024_corrTop25.csv"),
    Path("GEE_Exports_R3000/Table_ConfusionMatrix_RowNorm_Long_DakLak2024_corrTop25.csv"),
    Path("data/Table_ConfusionMatrix_RowNorm_Long_DakLak2024_corrTop25.csv"),
    Path("GEE_Exports_R3000/Table_ConfusionMatrix_Long_DakLak2024_corrTop25.csv"),
    Path("data/Table_ConfusionMatrix_Long_DakLak2024_corrTop25.csv"),
    Path("Supplementary/confusion_matrix_counts.csv"),
    Path("Figures/confusion_matrix_counts.csv"),
]

OUT_DIR = FIGURES_DIR

CLASS_NAMES = {
    1: "Sun coffee",
    2: "Intercrop coffee",
    3: "Newly planted coffee",
    4: "Rubber",
    5: "Partially vegetative",
    6: "Rice",
    7: "Other upland crops",
    8: "Forest",
    9: "Water",
    10: "Built",
}
CLASS_ORDER = list(CLASS_NAMES.keys())

plt.rcParams.update({
    "font.family": "Arial",
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


def _first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find confusion matrix input. Tried:\n  " + "\n  ".join(str(p) for p in paths))


def _col(df, candidates):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    return None


def load_matrix(path: Path):
    print(f"Loading: {path}")
    df = pd.read_csv(path)

    # GEE row-normalized long format.
    if {"true_class_id", "pred_class_id"}.issubset(df.columns) and (
        "row_normalized_percent" in df.columns or "row_normalized_proportion" in df.columns
    ):
        mat = np.zeros((len(CLASS_ORDER), len(CLASS_ORDER)), dtype=float)
        value_col = "row_normalized_proportion" if "row_normalized_proportion" in df.columns else "row_normalized_percent"
        for _, r in df.iterrows():
            t, p = int(r["true_class_id"]), int(r["pred_class_id"])
            v = float(r[value_col])
            if value_col == "row_normalized_percent":
                v /= 100.0
            if t in CLASS_ORDER and p in CLASS_ORDER:
                mat[CLASS_ORDER.index(t), CLASS_ORDER.index(p)] = v
        return mat, path, "row_normalized_long"

    # GEE/raw long format.
    if "count" in df.columns:
        true_col = _col(df, ["true_class_id", "true_class", "reference", "class_id", "actual"])
        pred_col = _col(df, ["pred_class_id", "pred_class", "predicted", "classification", "prediction"])
        if true_col is None or pred_col is None:
            raise ValueError(f"Could not detect long-format columns in {path}. Columns: {df.columns.tolist()}")
        counts = np.zeros((len(CLASS_ORDER), len(CLASS_ORDER)), dtype=float)
        for _, r in df.iterrows():
            t, p, c = int(r[true_col]), int(r[pred_col]), float(r["count"])
            if t in CLASS_ORDER and p in CLASS_ORDER:
                counts[CLASS_ORDER.index(t), CLASS_ORDER.index(p)] = c
        row_sums = counts.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        return counts / row_sums, path, "raw_long"

    # Square matrix.
    df_sq = pd.read_csv(path, index_col=0)
    df_sq.index = pd.to_numeric(df_sq.index, errors="coerce").astype("Int64")
    df_sq.columns = pd.to_numeric(df_sq.columns, errors="coerce").astype("Int64")
    df_sq = df_sq.reindex(index=CLASS_ORDER, columns=CLASS_ORDER).fillna(0.0)
    counts = df_sq.to_numpy(dtype=float)
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return counts / row_sums, path, "square_counts"


input_path = _first_existing(CANDIDATE_INPUTS)
cm_norm, src, fmt = load_matrix(input_path)
print(f"Format: {fmt}; shape: {cm_norm.shape}")

# Producer accuracy audit table.
pa = np.diag(cm_norm) * 100
pd.DataFrame({
    "class_id": CLASS_ORDER,
    "class_name": [CLASS_NAMES[i] for i in CLASS_ORDER],
    "producer_accuracy_percent": pa,
}).to_csv(OUT_DIR / "fig4_producer_accuracy.csv", index=False, encoding="utf-8-sig")

labels = [CLASS_NAMES[cid] for cid in CLASS_ORDER]
fig, ax = plt.subplots(figsize=(6.0, 6.0))

im = ax.imshow(cm_norm, cmap=plt.cm.Blues, norm=mcolors.Normalize(vmin=0, vmax=1), aspect="equal")

for i in range(len(CLASS_ORDER)):
    for j in range(len(CLASS_ORDER)):
        val = cm_norm[i, j]
        if val < 0.005:
            continue
        txt_color = "white" if val > 0.55 else "#1a1a1a"
        weight = "bold" if i == j else "regular"
        ax.text(j, i, f"{val * 100:.0f}", ha="center", va="center", fontsize=10, color=txt_color, fontweight=weight)

ax.set_xticks(range(len(CLASS_ORDER)))
ax.set_yticks(range(len(CLASS_ORDER)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted class")
ax.set_ylabel("Reference class")

for i in range(len(CLASS_ORDER)):
    ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="#444", linewidth=1.1, zorder=5))

ax.set_xticks(np.arange(len(CLASS_ORDER)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(CLASS_ORDER)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linewidth=0.5)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Row-normalized proportion")
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT_DIR / f"Figure5_ConfusionMatrix.{ext}", dpi=600 if ext == "png" else None, bbox_inches="tight")
plt.close(fig)
print("Saved Figure 4 outputs in", OUT_DIR.resolve())


Loading: data\raw\Table_ConfusionMatrix_Long_DakLak2024_corrTop25.csv
Format: raw_long; shape: (10, 10)


Saved Figure 4 outputs in D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\figures


In [12]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================
manifest_rows = []
for root in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                manifest_rows.append({
                    "folder": root.name,
                    "file": str(p.relative_to(root)),
                    "size_kb": round(p.stat().st_size / 1024, 1),
                })
manifest = pd.DataFrame(manifest_rows)
manifest_path = SUPPLEMENTARY_DIR / f"Manifest_{Path().resolve().name}.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(manifest.tail(30))
print("Manifest saved:", manifest_path)


,folder,file,size_kb
178,supplementary,S06_block_summary_20km.csv,3.3
179,supplementary,selected_features_by_fold.csv,23.4
180,supplementary,selected_features_used_by_RF_SHAP.csv,0.3
181,supplementary,selection_frequency.csv,9.3
182,supplementary,shap_class_10_importance.csv,2.0
183,supplementary,shap_class_1_importance.csv,2.1
184,supplementary,shap_class_2_importance.csv,2.3
185,supplementary,shap_class_3_importance.csv,2.3
186,supplementary,shap_class_4_importance.csv,2.0
187,supplementary,shap_class_5_importance.csv,2.3


Manifest saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\Manifest_mmlab-coffeemap-daklak.csv
